# Protein interface analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davismerighi-max/interface-analysis/blob/main/notebooks/interface_analysis_colab.ipynb)

Give it a refined structure (`.pdb` / `.cif`, or a PDB entry code) and it measures every
interface: which residues are buried and by how much, the salt bridges, hydrogen bonds,
hydrophobic contacts, disulfides, metal bridges and stacking that hold the subunits together,
how well the surfaces fit, what bound cofactors touch, and which parts of the model could be
stable assemblies on their own. Then it draws the figures and hands you everything as a zip.

### How to use it — no programming needed

1. **Press ▶ on box ①.** A form opens below it: the structure (a PDB entry code, or
   *Upload* your own `.pdb` / `.cif`), optionally the name, colour and sequence of each protein
   (a new line appears as you fill one), alignments for conservation, and the options.
   Filling in the form runs nothing.
2. **Press ▶ on box ②.** It runs every step and shows the results under it; a zip with
   everything, sorted into one folder per figure, downloads at the end.

To try it, type `2TSC` as the PDB code. *Runtime → Run all* is safe too: box ② then only
reminds you to fill in the form.

Code and documentation: https://github.com/davismerighi-max/interface-analysis

In [ ]:
#@title ① Settings — press ▶, then fill in the form { display-mode: "form" }
#@markdown Press ▶ on the left to open the form. Nothing is analysed while you fill it in: the run starts only when you press ▶ on box ②. (Meanwhile the libraries and the code are fetched in the background.)
import os, sys, subprocess, ipywidgets as W
from IPython.display import display

CONTENT = "/content"
REPO = CONTENT + "/interface-analysis"
if "_SETUP" not in globals() and not os.path.exists(REPO + "/01_interface_analysis.py"):
    _SETUP = subprocess.Popen(
        f'"{sys.executable}" -m pip install -q biopython pydssp scikit-image && '
        f"git clone --depth 1 -q https://github.com/davismerighi-max/interface-analysis.git {REPO}",
        shell=True, stdout=open(os.path.join(CONTENT, "setup.log"), "w"),
        stderr=subprocess.STDOUT)


def _uploaded(w):
    # [(name, bytes)] from a FileUpload widget, for ipywidgets 7 and 8 alike
    v = w.value
    if isinstance(v, dict):
        return [(n, bytes(d["content"])) for n, d in v.items()]
    return [(d["name"], bytes(d["content"])) for d in v]


def _reset(w):
    # empty the upload button once its file is taken, so the same file can be picked again
    try:
        w.value = {} if isinstance(w.value, dict) else ()
        if hasattr(w, "_counter"):
            w._counter = 0
    except Exception:
        pass


def _h(title, sub=""):
    return W.HTML(f"<div style='margin:16px 0 3px;font-weight:600;font-size:15px'>{title}</div>"
                  + (f"<div style='color:#666;font-size:12px;max-width:860px'>{sub}</div>"
                     if sub else ""))


_ST = {"description_width": "initial"}
if "_F" not in globals():        # re-running this box keeps what you already typed
    _F = {}
    SOURCE, FASTA, MSAS = {}, {}, {}

    # ---- structure
    _F["code"] = W.Text(placeholder="e.g. 2TSC", description="PDB code", style=_ST,
                        layout=W.Layout(width="230px"))
    _F["up"] = W.FileUpload(accept=".pdb,.cif,.ent,.mmcif", multiple=False,
                            description="Upload my structure", layout=W.Layout(width="200px"))
    _F["status"] = W.HTML()
    _F["res"] = W.FloatText(value=0, description="resolution (Å)", style=_ST,
                            layout=W.Layout(width="190px"))

    def _show_source():
        c = _F["code"].value.strip()
        if SOURCE:
            _F["status"].value = (f"✔ will use your uploaded file <b>{SOURCE['name']}</b> "
                                  f"({len(SOURCE['data']) / 1e6:.1f} MB)")
        elif c.lower().endswith((".pdb", ".cif", ".ent", ".mmcif")):
            _F["status"].value = f"✔ will use the file <b>{c}</b> from the Files panel"
        elif c:
            _F["status"].value = f"✔ will download PDB entry <b>{c.upper()}</b>"
        else:
            _F["status"].value = ("<span style='color:#B33A3A'>type a PDB code or upload "
                                  "a file</span>")

    def _on_code(ch):
        if ch["new"].strip():
            SOURCE.clear()
        _show_source()

    def _on_up(ch):
        f = _uploaded(_F["up"])
        if f:
            SOURCE.clear(); SOURCE.update(name=f[-1][0], data=f[-1][1])
            _F["code"].unobserve(_on_code, "value"); _F["code"].value = ""
            _F["code"].observe(_on_code, "value")
            _reset(_F["up"])
        _show_source()

    _F["code"].observe(_on_code, "value"); _F["up"].observe(_on_up, "value"); _show_source()

    # ---- proteins: name | colour | sequence; a new empty line appears when the last is used
    _PAL = ["#7FA8D4", "#E8884A", "#95C892", "#B48BC4", "#D4B36A", "#7FC4C4", "#D98FA6",
            "#A3A3A3"]
    _F["rows"] = W.VBox()

    def _filled(row):
        return bool(row.children[0].value.strip() or row.children[2].value.strip())

    def _new_row():
        k = len(_F["rows"].children)
        name = W.Text(placeholder="name", layout=W.Layout(width="150px"))
        col = W.ColorPicker(value=_PAL[k % len(_PAL)], concise=False,
                            layout=W.Layout(width="150px"))
        seq = W.Text(placeholder="sequence (one-letter code, paste the whole chain)",
                     layout=W.Layout(flex="1 1 auto", min_width="320px"))
        rm = W.Button(icon="times", tooltip="remove this protein", layout=W.Layout(width="38px"))
        row = W.HBox([name, col, seq, rm], layout=W.Layout(width="100%", max_width="1100px"))

        def _grow(ch):
            kids = _F["rows"].children
            if kids and kids[-1] is row and _filled(row) and len(kids) < 40:
                _new_row()

        def _rm(_):
            kids = [r for r in _F["rows"].children if r is not row]
            if not kids:
                name.value = ""; seq.value = ""
                return
            _F["rows"].children = tuple(kids)
            if _filled(kids[-1]):
                _new_row()

        name.observe(_grow, "value"); seq.observe(_grow, "value"); rm.on_click(_rm)
        _F["rows"].children = _F["rows"].children + (row,)

    _new_row()
    _F["fasta"] = W.FileUpload(accept=".fasta,.fa,.faa,.txt", multiple=False,
                               description="or upload a FASTA", layout=W.Layout(width="190px"))
    _F["fclear"] = W.Button(description="use the lines instead", layout=W.Layout(width="170px"))
    _F["pstatus"] = W.HTML()

    def _show_fasta():
        _F["pstatus"].value = (f"✔ will use <b>{FASTA['name']}</b>; the lines above are ignored"
                               if FASTA else
                               "the lines above are used (empty lines are skipped)")

    def _on_fasta(ch):
        f = _uploaded(_F["fasta"])
        if f:
            FASTA.clear(); FASTA.update(name=f[-1][0], data=f[-1][1]); _reset(_F["fasta"])
        _show_fasta()

    _F["fasta"].observe(_on_fasta, "value")
    _F["fclear"].on_click(lambda _: (FASTA.clear(), _show_fasta()))
    _show_fasta()

    # ---- alignments
    _F["msa"] = W.FileUpload(accept=".fasta,.fa,.a3m,.aln,.sto,.stockholm,.clustal,.txt",
                             multiple=True, description="Upload alignment(s)",
                             layout=W.Layout(width="210px"))
    _F["mclear"] = W.Button(description="remove alignments", layout=W.Layout(width="170px"))
    _F["mstatus"] = W.HTML()

    def _show_msa():
        _F["mstatus"].value = ("✔ " + ", ".join(f"<b>{n}</b>" for n in MSAS)) if MSAS else \
            "no alignment: the footprints are drawn without a conservation track"

    def _on_msa(ch):
        f = _uploaded(_F["msa"])
        for n, d in f:
            MSAS[n] = d
        if f:
            _reset(_F["msa"])
        _show_msa()

    _F["msa"].observe(_on_msa, "value")
    _F["mclear"].on_click(lambda _: (MSAS.clear(), _show_msa()))
    _show_msa()

    # ---- options
    def _box(v, d):
        return W.Checkbox(value=v, description=d, indent=False, style=_ST,
                          layout=W.Layout(width="auto", margin="0 28px 0 0"))

    _F["shape"] = _box(True, "shape complementarity (the slow part)")
    _F["assem"] = _box(True, "find candidate assemblies")
    _F["perline"] = W.BoundedIntText(value=350, min=50, max=5000, description="residues per line",
                                     style=_ST, layout=W.Layout(width="210px"))
    _F["render"] = _box(True, "draw the 3D renders")
    _F["elev"] = W.BoundedFloatText(value=62, min=0, max=90, description="tilted view: elevation",
                                    style=_ST, layout=W.Layout(width="230px"))
    _F["azim"] = W.FloatText(value=20, description="rotation", style=_ST,
                             layout=W.Layout(width="150px"))
    _F["again"] = _box(True, "measure again")
    _F["zip"] = _box(True, "download the zip at the end")


def settings():
    # every value of the form, read when box ② starts
    rows = []
    for r in _F["rows"].children:
        rows.append((r.children[0].value, r.children[1].value, r.children[2].value))
    return dict(code=_F["code"].value.strip(), resolution=_F["res"].value, proteins=rows,
                shape=_F["shape"].value, assemblies=_F["assem"].value,
                per_line=_F["perline"].value, render=_F["render"].value,
                elev=_F["elev"].value, azim=_F["azim"].value,
                again=_F["again"].value, zip=_F["zip"].value)


_hdr = W.HTML("<div style='display:flex;color:#666;font-size:12px'>"
              "<span style='width:150px'>name</span><span style='width:150px'>colour</span>"
              "<span>sequence</span></div>")
display(W.VBox([
    _h("Structure", "Type a PDB entry code <b>or</b> upload your model. A file you already "
       "dropped into the Files panel on the left can be typed instead of a code, e.g. "
       "<code>my_model.cif</code> — the better route for files above ~50 MB."),
    W.HBox([_F["code"], W.HTML("<span style='margin:0 14px;color:#777'>or</span>"), _F["up"]]),
    _F["status"],
    _F["res"],
    W.HTML("<div style='color:#666;font-size:12px'>resolution of the map or data; 0 = read it "
           "from the file header (if it is not there either, bonds are reported as "
           "tentative)</div>"),
    _h("Proteins (optional)", "Proteins are recognised in the model <b>by sequence</b>, so "
       "order does not matter and the full-length sequence is fine; a new line appears as you "
       "fill the last one. Anything that does not match runs with default names and colours, "
       "and the run report says why. A FASTA file works too (headers like "
       "<code>&gt;YfdQ_#FF00F7</code>)."),
    _hdr, _F["rows"], W.HBox([_F["fasta"], _F["fclear"]]), _F["pstatus"],
    _h("Conservation (optional)", "One alignment per protein: aligned FASTA, A3M "
       "(ColabFold / HHblits), Clustal or Stockholm; the <b>first</b> sequence must be your "
       "protein. Each one is matched to its protein by sequence."),
    W.HBox([_F["msa"], _F["mclear"]]), _F["mstatus"],
    _h("Analysis", "Shape complementarity takes about a minute for a dimer and ten minutes or "
       "more for ~30 chains; untick it for a quick first look (the packing figure is then "
       "left out)."),
    W.HBox([_F["shape"], _F["assem"]]),
    _h("Figures", "The 3D renders show three views: side, tilted and top. Elevation 0 = side, "
       "90 = top; rotation turns the model about its symmetry axis."),
    W.HBox([_F["perline"], _F["render"]]),
    W.HBox([_F["elev"], _F["azim"]]),
    _h("Re-running", "Untick <i>measure again</i> to redraw the figures of the SAME model with "
       "new figure settings without measuring again (much faster)."),
    W.HBox([_F["again"], _F["zip"]]),
    W.HTML("<div style='margin-top:16px;padding:8px 12px;background:#EEF4FB;border-radius:6px;"
           "max-width:860px'>When the form is ready, press <b>▶ on box ②</b> below.</div>"),
]))

In [ ]:
#@title ② Run everything { display-mode: "form" }
#@markdown Press ▶ once the form in box ① is ready. This box has no settings of its own.
import os, sys, re, time, glob, shutil, subprocess
from IPython.display import display, Image, Markdown

try:
    from google.colab import files as _colab_files
except Exception:
    _colab_files = None


def _step(label, script, args):
    # run one step: one progress line, and the end of its log only if it stops
    t0 = time.time()
    print(f"{label} ...", end=" ", flush=True)
    p = subprocess.run([sys.executable, f"{REPO}/{script}", *args],
                       capture_output=True, text=True)
    if p.returncode:
        print(f"STOPPED after {time.time() - t0:.0f} s")
        print("\n".join((p.stdout + p.stderr).splitlines()[-25:]))
    else:
        print(f"done ({time.time() - t0:.0f} s)")
    return p.returncode


def _show(paths, title):
    paths = [p for p in paths if os.path.exists(p)]
    if paths:
        if title:
            display(Markdown("## " + title))
        for p in paths:
            display(Markdown(f"`{p}`"))
            display(Image(filename=p, width=1000))


def _run():
    global MODEL, RUN
    if "_F" not in globals():
        print("Open the form first: press ▶ on box ①, fill it in, then press ▶ here.")
        return
    S = settings()
    if not SOURCE and not S["code"]:
        print("Nothing to run yet: in box ①, type a PDB code or upload a structure, "
              "then press ▶ on this box.")
        return
    os.chdir(CONTENT)

    # ---- libraries and code (box ① started fetching them in the background)
    if "_SETUP" in globals() and _SETUP.poll() is None:
        print("finishing the installation ...", flush=True)
        _SETUP.wait()
    if not os.path.exists(REPO + "/01_interface_analysis.py"):
        print("installing libraries and fetching the code (about a minute) ...", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "biopython", "pydssp", "scikit-image"], check=True)
        if os.path.isdir(REPO):
            shutil.rmtree(REPO)
        subprocess.run(["git", "clone", "--depth", "1", "-q",
                        "https://github.com/davismerighi-max/interface-analysis.git", REPO],
                       check=True)

    # ---- the structure
    c = S["code"]
    if SOURCE:
        MODEL = os.path.basename(SOURCE["name"]).replace(" ", "_")
        data = SOURCE["data"]
    elif c.lower().endswith((".pdb", ".cif", ".ent", ".mmcif")):       # from the Files panel
        p = c if os.path.isabs(c) else os.path.join(CONTENT, c)
        if not os.path.exists(p):
            print(f"Box ①: there is no file called {c} in the Files panel.")
            return
        MODEL = os.path.basename(p).replace(" ", "_")
        data = open(p, "rb").read()
    else:
        import urllib.request
        data = None
        for ext in (".pdb", ".cif"):               # very large entries exist only as mmCIF
            try:
                with urllib.request.urlopen(
                        f"https://files.rcsb.org/download/{c.lower()}{ext}") as r:
                    data = r.read()
                MODEL = c.lower() + ext
                break
            except Exception:
                continue
        if data is None:
            print(f"Could not download PDB entry {c.upper()}: check the code in box ①.")
            return

    # each model works in its own folder, so results of different models never mix
    RUN = os.path.join(CONTENT, "runs", os.path.splitext(MODEL)[0])
    reuse = (not S["again"]) and os.path.exists(os.path.join(RUN, "run_config.json"))
    if not reuse:
        if os.path.isdir(RUN):
            shutil.rmtree(RUN)
        os.makedirs(RUN)
        open(os.path.join(RUN, MODEL), "wb").write(data)
    os.chdir(RUN)
    print("model:", MODEL, "| working folder:", RUN, "| measurements reused" if reuse else "")

    # ---- proteins -> proteins.fasta
    proteins = None
    if FASTA:
        open("proteins.fasta", "wb").write(FASTA["data"])
        proteins = "proteins.fasta"
    else:
        out = []
        for i, (name, col, seq) in enumerate(S["proteins"], 1):
            name = "_".join(name.split())
            seq = re.sub(r"[^A-Za-z]", "", seq).upper()
            if name and seq:
                out.append(f">{name}_{col}\n{seq}\n")
            elif name or seq:
                print(f"protein line {i}: needs both a name and a sequence - skipped")
        if out:
            open("proteins.fasta", "w").write("".join(out))
            proteins = "proteins.fasta"

    # ---- alignments
    msas = []
    for i, (n, d) in enumerate(MSAS.items(), 1):
        safe = "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in os.path.basename(n))
        open(f"msa_{i}_{safe}", "wb").write(d)
        msas.append(f"msa_{i}_{safe}")

    # ---- the steps
    ok = True
    if not reuse:
        a1 = [MODEL]
        if S["resolution"] and S["resolution"] > 0:
            a1 += ["--resolution", f"{S['resolution']:g}"]
        a1 += ["--shape-workers", "2"] if S["shape"] else ["--no-shape"]
        if proteins:
            a1 += ["--proteins", proteins]
        ok = _step("1/5  measuring the interfaces", "01_interface_analysis.py", a1) == 0
        ok = ok and _step("2/5  tables and statistics", "02_derive_tables.py", [MODEL]) == 0
    if ok:
        a3 = [MODEL, "--per-line", str(S["per_line"])]
        a3 += (["--render-view", f"{S['elev']:g},{S['azim']:g}"] if S["render"]
               else ["--no-render"])
        if msas:
            a3 += ["--msa", ",".join(msas)]
        _step("3/5  figures", "03_make_figures.py", a3)
        if S["assemblies"]:
            _step("4/5  candidate assemblies", "05_assemblies.py",
                  [MODEL] + (["--render"] if S["render"] else []))
        _step("5/5  sorting the results", "06_organise_results.py", [MODEL])

    # ---- results
    if os.path.exists("run_report.txt"):
        display(Markdown("## Run report"))
        print(open("run_report.txt", encoding="utf-8").read())
    if not ok:
        print("A step stopped: the report above says which and why (details in run_errors.log).")
        return
    _show(["fig_render_proteins.png", "fig_render_budget.png", "fig_render_interactions.png"],
          "3D renders")
    _show(sorted(glob.glob("fig*_footprints.png")), "Footprints")
    _show(["fig_inventory.png", "fig_area_partition.png", "fig_chemistry.png", "fig_packing.png",
           "fig_architecture.png", "fig_contact_maps.png", "figS1_contact_maps_all.png"]
          + sorted(glob.glob("fig_interaction_networks*.png")), "Interfaces")
    if os.path.exists("assembly_candidates.csv"):
        import pandas as pd
        pd.set_option("display.max_colwidth", 200)
        display(Markdown("## Candidate assemblies"))
        display(pd.read_csv("assembly_candidates.csv")[
            ["rank", "class", "kept_pct", "n_chains", "composition",
             "disulfides_inside", "disulfides_across_boundary", "reason"]])
        _show(["fig_assemblies.png"], "")
    base = os.path.splitext(MODEL)[0] + "_results"
    if os.path.isdir(base):
        z = shutil.make_archive(os.path.join(CONTENT, base), "zip", RUN, base)
        print("\nall results, one folder per figure:", z)
        if S["zip"] and _colab_files is not None:
            _colab_files.download(z)


_run()

---
### Reading three of the numbers

**Buried area** is computed pairwise in isolation and reported halved (the interface area,
not the sum of both footprints): PISA's convention, so directly comparable to it.

**Shape complementarity** reads about **0.10 below** the original `sc` program (trypsin–BPTI
0.647; obligate histone heterodimers 0.598–0.620; published ≈ 0.70), so the class names on the
figure are pessimistic. Use S_c to rank interfaces within one run.

**ΔG** is the Eisenberg–McLachlan solvation term (20 amino acids only). Negative = hydrophobic
burial pays; positive = the interface is held by hydrogen bonds and salt bridges instead. It is
not a binding free energy.

### If something goes wrong
Read the run report under box ②: each item says *ok*, *DEFAULT*, *WARNING*, *SKIPPED* or
*FAILED*, with the reason. A failure in one figure never stops the others. Very large
assemblies may exceed the free Colab machine's memory — run them locally or on a cluster; the
code is identical.